---

### Python Workshop Part 8: More Pandas

© Kerry Back, Rice University

---

### Sorting Data

Pandas provides several methods to sort your data.

In [15]:
# Run this cell
import pandas as pd

# Create sample employee data
employees = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'department': ['Sales', 'Engineering', 'Sales', 'Marketing', 'Engineering'],
    'salary': [50000, 80000, 55000, 60000, 85000],
    'years_experience': [2, 5, 3, 4, 6]
})

print("Original data:")
print(employees)
print()

# Sort by salary (ascending)
print("Sorted by salary (ascending):")
print(employees.sort_values('salary'))
print()

# Sort by salary (descending)
print("Sorted by salary (descending):")
print(employees.sort_values('salary', ascending=False))

Original data:
      name   department  salary  years_experience
0    Alice        Sales   50000                 2
1      Bob  Engineering   80000                 5
2  Charlie        Sales   55000                 3
3    Diana    Marketing   60000                 4
4      Eve  Engineering   85000                 6

Sorted by salary (ascending):
      name   department  salary  years_experience
0    Alice        Sales   50000                 2
2  Charlie        Sales   55000                 3
3    Diana    Marketing   60000                 4
1      Bob  Engineering   80000                 5
4      Eve  Engineering   85000                 6

Sorted by salary (descending):
      name   department  salary  years_experience
4      Eve  Engineering   85000                 6
1      Bob  Engineering   80000                 5
3    Diana    Marketing   60000                 4
2  Charlie        Sales   55000                 3
0    Alice        Sales   50000                 2


In [16]:
# Run this cell - sorting by multiple columns
print("Sorted by department, then by salary (descending):")
print(employees.sort_values(['department', 'salary'], ascending=[True, False]))
print()

# Sort by index
shuffled = employees.sample(frac=1)  # Randomly shuffle rows
print("Shuffled data:")
print(shuffled)
print()
print("Sorted by index:")
print(shuffled.sort_index())

Sorted by department, then by salary (descending):
      name   department  salary  years_experience
4      Eve  Engineering   85000                 6
1      Bob  Engineering   80000                 5
3    Diana    Marketing   60000                 4
2  Charlie        Sales   55000                 3
0    Alice        Sales   50000                 2

Shuffled data:
      name   department  salary  years_experience
2  Charlie        Sales   55000                 3
3    Diana    Marketing   60000                 4
4      Eve  Engineering   85000                 6
1      Bob  Engineering   80000                 5
0    Alice        Sales   50000                 2

Sorted by index:
      name   department  salary  years_experience
0    Alice        Sales   50000                 2
1      Bob  Engineering   80000                 5
2  Charlie        Sales   55000                 3
3    Diana    Marketing   60000                 4
4      Eve  Engineering   85000                 6


### Grouping and Aggregation

Grouping allows you to split your data into groups based on some criteria and then apply functions to each group.  

Some standard functions that you can apply to each group are mean, median, min, max, std (std deviation), and quantile.

You can also apply your own custom functions to each group.

In [17]:
# Run this cell - basic grouping
print("Average salary by department:")
dept_avg = employees.groupby('department')['salary'].mean()
print(dept_avg)
print()

print("Maximum salary by department:")
dept_stats = employees.groupby('department')['salary'].max()
print(dept_stats)
print()

# Transform a numeric variable into categories
employees['experience_level'] = pd.cut(employees['years_experience'], 
                                     bins=[0, 3, 5, 10], 
                                     labels=['Junior', 'Mid', 'Senior'])

# Group by multiple columns
finer_classification = employees.groupby(
    ['department', 'experience_level'],
    observed=False).size()
print('Group by multiple columns')
print(finer_classification)

Average salary by department:
department
Engineering    82500.0
Marketing      60000.0
Sales          52500.0
Name: salary, dtype: float64

Maximum salary by department:
department
Engineering    85000
Marketing      60000
Sales          55000
Name: salary, dtype: int64

Group by multiple columns
department   experience_level
Engineering  Junior              0
             Mid                 1
             Senior              1
Marketing    Junior              0
             Mid                 1
             Senior              0
Sales        Junior              2
             Mid                 0
             Senior              0
dtype: int64


### MultiIndexes

The last example illustrated the pandas function `pd.cut`, which creates a categorization based on the value of a numeric variable.  It is a shorthand if-elif-else type of function.

The example also illustrates another new concept, **MultiIndexes**.  The finer_classification DataFrame has two levels of indexing.  MultiIndexes can be created whenever they are useful.  They are created automatically when grouping by multiple variables.

We would normally want to make this DataFrame easier to read by turning one of the indexes into column labels, creating a **pivot table** from the original data.  We can do this with `unstack`.

In [18]:
# Run this cell

finer_classification_unstacked = finer_classification.unstack()
print('Count by experience level and department')
print(finer_classification_unstacked)

Count by experience level and department
experience_level  Junior  Mid  Senior
department                           
Engineering            0    1       1
Marketing              0    1       0
Sales                  2    0       0


### More Pivot Tables

Pandas provides a `pivot_table` function that is an alternative to grouping by multiple variables and unstacking.  If what we really want is a pivot table, then it is easier to use it.

In [19]:
# Run this cell to see pivot tables in action

print("=== PIVOT TABLE EXAMPLES ===")

# Create sales data for pivot table examples
import random
random.seed(42)

sales = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=30).repeat(3),
    'product': ['Widget A', 'Widget B', 'Widget C'] * 30,
    'region': (['North'] * 45) + (['South'] * 45),
    'quantity': [random.randint(10, 50) for _ in range(90)],
    'revenue': [random.randint(100, 500) for _ in range(90)]
})

print("Sample sales data (first 10 rows):")
print(sales.head(10))
print(f"Total rows: {len(sales)}")

# 1. Basic pivot table - average quantity by product and region
print("\n1. Average quantity by product and region:")
pivot1 = sales.pivot_table(values='quantity', 
                           index='product', 
                           columns='region', 
                           aggfunc='mean')
print(pivot1)

# 2. Multiple aggregations
print("\n2. Multiple aggregations (sum and mean):")
pivot2 = sales.pivot_table(values='revenue', 
                           index='product', 
                           columns='region', 
                           aggfunc=['sum', 'mean', 'count'])
print(pivot2)

# 3. Multiple values
print("\n3. Pivot with multiple values:")
pivot3 = sales.pivot_table(values=['quantity', 'revenue'], 
                           index='product', 
                           columns='region', 
                           aggfunc='sum')
print(pivot3)

# 4. Add margins (totals)
print("\n4. Pivot table with totals:")
pivot4 = sales.pivot_table(values='revenue', 
                           index='product', 
                           columns='region', 
                           aggfunc='sum',
                           margins=True,
                           margins_name='Total')
print(pivot4)

=== PIVOT TABLE EXAMPLES ===
Sample sales data (first 10 rows):
        date   product region  quantity  revenue
0 2024-01-01  Widget A  North        50      116
1 2024-01-01  Widget B  North        17      261
2 2024-01-01  Widget C  North        11      305
3 2024-01-02  Widget A  North        27      237
4 2024-01-02  Widget B  North        25      133
5 2024-01-02  Widget C  North        24      208
6 2024-01-03  Widget A  North        18      390
7 2024-01-03  Widget B  North        16      467
8 2024-01-03  Widget C  North        44      261
9 2024-01-04  Widget A  North        15      208
Total rows: 90

1. Average quantity by product and region:
region        North      South
product                       
Widget A  26.800000  32.333333
Widget B  24.666667  32.866667
Widget C  30.466667  26.800000

2. Multiple aggregations (sum and mean):
           sum              mean             count      
region   North South       North       South North South
product                    

### Data Cleaning

Real-world data is often messy. Let's learn how to handle common data quality issues.

In [20]:
# Run this cell - handling missing data

import numpy as np

# Create data with missing values
messy_data = pd.DataFrame({
    'product': ['Widget A', 'Widget B', None, 'Widget C', 'Widget D'],
    'price': [10.5, None, 15.0, 12.75, None],
    'quantity': [100, 200, 150, None, 300],
    'category': ['Electronics', 'Electronics', 'Home', 'Electronics', 'Home']
})

print("Data with missing values:")
print(messy_data)
print()

# Check for missing values
# .isnull returns a boolean
# When we sum booleans, True is converted to 1 and False to 0
print("Missing values per column:")
print(messy_data.isnull().sum())
print()

# We might want to drop rows with any missing values
print("After dropping rows with missing values:")
dropped_data = messy_data.dropna()
print(dropped_data)
print()

# We might want to fill missing values
# We can choose what to fill them with
print("After filling missing values:")
filled_data = messy_data.copy()
filled_data['product'] = filled_data['product'].fillna('Unknown')
mean_price = messy_data['price'].mean()
filled_data['price'] = filled_data['price'].fillna(mean_price)
filled_data['quantity']=  filled_data['quantity'].fillna(0)
print(filled_data)

Data with missing values:
    product  price  quantity     category
0  Widget A  10.50     100.0  Electronics
1  Widget B    NaN     200.0  Electronics
2      None  15.00     150.0         Home
3  Widget C  12.75       NaN  Electronics
4  Widget D    NaN     300.0         Home

Missing values per column:
product     1
price       2
quantity    1
category    0
dtype: int64

After dropping rows with missing values:
    product  price  quantity     category
0  Widget A   10.5     100.0  Electronics

After filling missing values:
    product  price  quantity     category
0  Widget A  10.50     100.0  Electronics
1  Widget B  12.75     200.0  Electronics
2   Unknown  15.00     150.0         Home
3  Widget C  12.75       0.0  Electronics
4  Widget D  12.75     300.0         Home


### Merging and Joining DataFrames

We often need to combine data from multiple sources. Pandas provides several powerful methods to merge and join DataFrames.

We can implement any of the different types of merges (inner join, left join, right join, outer join).  It is easiest to explain these through an example.

### Inner, Left, Right, and Outer Joins

In [21]:
# Run this cell to create example data

# Employee information
employees_info = pd.DataFrame({
    'employee_id': [101, 102, 103, 104, 105],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'department': ['Sales', 'Engineering', 'Sales', 'Marketing', 'Engineering']
})

# Employee salaries (some employees missing)
salaries = pd.DataFrame({
    'employee_id': [101, 102, 104, 106],  # Note: 103, 105 missing, 106 is new
    'salary': [50000, 80000, 60000, 75000],
    'bonus': [5000, 10000, 6000, 8000]
})

**Types of joins**

Suppose we want to look at salary data by department.  To do so, we need to merge Employee Info with Salaries, matching employee_id's in the two DataFrames.  

Notice that employee_id = 105 is in the Employee Info table but not in Salaries.  And employee_id = 106 is in Salaries but not in Employee Info.  Do we want to include 105 and/or 106 in the merged table?

That depends on what we're planning to do.  What is important here is that there are four possibilities:

1. Include both 105 and 106
2. Include 105 but not 106
3. Include 106 but not 105
4. Don't include either.

#1 is called an outer join (include all records that are in at least one database).

#2 is called an inner join (include only records that are in both databases)

Depending on the order in which we list the DataFrames in the merge statement, #2 and #3 would be left joins or right joins.  For example, if Employee Info is first (meaning left) then #2 is a left join and #3 is a right join.

In [22]:
print("\n=== MERGE EXAMPLES ===")

# 1. Inner merge (default) - only matching records
print("\n1. Inner Merge (only employees with salary data):")
inner_merge = pd.merge(employees_info, salaries, on='employee_id')
print(inner_merge)

# 2. Left merge - all employees, with salary if available
print("\n2. Left Merge (all employees, salary if available):")
left_merge = pd.merge(employees_info, salaries, on='employee_id', how='left')
print(left_merge)

# 3. Right merge - all salary records
print("\n3. Right Merge (all salary records):")
right_merge = pd.merge(employees_info, salaries, on='employee_id', how='right')
print(right_merge)

# 4. Outer merge - all records from both
print("\n4. Outer Merge (all records from both):")
outer_merge = pd.merge(employees_info, salaries, on='employee_id', how='outer')
print(outer_merge)


=== MERGE EXAMPLES ===

1. Inner Merge (only employees with salary data):
   employee_id   name   department  salary  bonus
0          101  Alice        Sales   50000   5000
1          102    Bob  Engineering   80000  10000
2          104  Diana    Marketing   60000   6000

2. Left Merge (all employees, salary if available):
   employee_id     name   department   salary    bonus
0          101    Alice        Sales  50000.0   5000.0
1          102      Bob  Engineering  80000.0  10000.0
2          103  Charlie        Sales      NaN      NaN
3          104    Diana    Marketing  60000.0   6000.0
4          105      Eve  Engineering      NaN      NaN

3. Right Merge (all salary records):
   employee_id   name   department  salary  bonus
0          101  Alice        Sales   50000   5000
1          102    Bob  Engineering   80000  10000
2          104  Diana    Marketing   60000   6000
3          106    NaN          NaN   75000   8000

4. Outer Merge (all records from both):
   employee_i

### Concatenating

Sometimes we have multiple datasets containing the same variables (for example, observations from multiple years) that we want to combine.  This is called concatenating.

In [23]:
# Run this cell.

# Create sample data for concatenation
q1_sales = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Widget C'],
    'sales': [100, 150, 200],
    'quarter': ['Q1', 'Q1', 'Q1']
})

q2_sales = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Widget C'],
    'sales': [120, 140, 220],
    'quarter': ['Q2', 'Q2', 'Q2']
})

q3_sales = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Widget C'],
    'sales': [110, 160, 210],
    'quarter': ['Q3', 'Q3', 'Q3']
})

print("Q1 Sales:")
print(q1_sales)
print("\nQ2 Sales:")
print(q2_sales)
print("\nQ3 Sales:")
print(q3_sales)

all_sales = pd.concat([q1_sales, q2_sales, q3_sales])
print("\nAll Sales:")
print(all_sales)


Q1 Sales:
    product  sales quarter
0  Widget A    100      Q1
1  Widget B    150      Q1
2  Widget C    200      Q1

Q2 Sales:
    product  sales quarter
0  Widget A    120      Q2
1  Widget B    140      Q2
2  Widget C    220      Q2

Q3 Sales:
    product  sales quarter
0  Widget A    110      Q3
1  Widget B    160      Q3
2  Widget C    210      Q3

All Sales:
    product  sales quarter
0  Widget A    100      Q1
1  Widget B    150      Q1
2  Widget C    200      Q1
0  Widget A    120      Q2
1  Widget B    140      Q2
2  Widget C    220      Q2
0  Widget A    110      Q3
1  Widget B    160      Q3
2  Widget C    210      Q3


### Reading from and Writing to Files

Pandas makes it easy to read data from various file formats and save your processed data back to files.

Pandas can read Excel worksheets.  It is simplest if each worksheet contains a single table.  

Pandas can also read CSV (comma separated) files and many other formats.  CSV files are text files and can be opened in NotePad, but if you double-click one it will probably open in Excel.  Excel can also Save As to CSV format.  

As an example, we will read an online Excel file.  To read from an Excel file on your hard drive, you would replace `url` with the filepath and filename of your file.

This online file does not have column names.  We're going to add column names separately.

The variable definitions are in this file: [https://faculty.utrgv.edu/diego.escobari/teaching/Datasets/WAGE1.txt](https://faculty.utrgv.edu/diego.escobari/teaching/Datasets/WAGE1.txt).

In [24]:
url = "https://faculty.utrgv.edu/diego.escobari/teaching/Datasets/WAGE1.xls"
wages = pd.read_excel(url, header=None)

columns = [
    'wage', 'educ', 'exper',
    'tenure', 'nonwhite', 'female',
    'married', 'numdep',
    'smsa', 'northcen',
    'south', 'west', 'construc',
    'ndurman', 'trcommpu', 'trade',
    'services', 'profserv',
    'profocc', 'clerocc', 'servocc',
    'lwage', 'expersq', 'tenursq'
]
wages.columns = columns


### Exercise 1: Exploring Data

In [25]:
# Get information about the wages DataFrame using methods you've seen earlier.  
# In particular, examine the first few rows to see its structure.  
# If you've forgotten how to do this, review or ask Gemini.




### Saving Data

Now, save the wages DataFrame to a CSV or Excel file on your Google Drive.  Ask Gemini to mount your Google Drive and ask Gemini to save the file.

Using your laptop's File Explorer, verify that the file was saved.

### String Processing

Text data often needs to be cleaned up.  Pandas provides powerful string methods through the `.str` accessor, allowing vectorized string operations on entire columns.

**Common String Operations:**
- **`.str.lower()`, `.str.upper()`** - Case conversion
- **`.str.strip()`** - Remove whitespace
- **`.str.contains()`** - Check for substring
- **`.str.split()`** - Split strings
- **`.str.replace()`** - Replace text
- **`.str.extract()`** - Extract pattern using regex

In [26]:
# Run this cell to see string processing operations

print("=== STRING PROCESSING EXAMPLES ===")

# Create sample data with messy strings
customer_data = pd.DataFrame({
    'customer_id': [1001, 1002, 1003, 1004, 1005, 1006],
    'name': ['  John Smith  ', 'JANE DOE', 'Bob Johnson', 'alice brown', 'Charlie Wilson', 'diana prince'],
    'email': ['john.smith@email.com', 'JANE@GMAIL.COM', 'bob@yahoo.com', 'alice@email.com', 'charlie@outlook.com', 'diana@email.com'],
    'phone': ['(555) 123-4567', '555.234.5678', '555-345-6789', '(555)456-7890', '555 567 8901', '555.678.9012'],
    'address': ['123 Main St, New York, NY', '456 Oak Ave, Boston, MA', '789 Pine Rd, Chicago, IL', 
                '321 Elm St, Houston, TX', '654 Maple Dr, Phoenix, AZ', '987 Cedar Ln, Philadelphia, PA']
})

print("Original customer data:")
print(customer_data)

# 1. Clean up names - strip whitespace and title case
print("\n1. Clean names:")
customer_data['name_clean'] = customer_data['name'].str.strip().str.title()
print(customer_data[['name', 'name_clean']])

# 2. Standardize email addresses to lowercase
print("\n2. Standardize emails:")
customer_data['email_clean'] = customer_data['email'].str.lower()
print(customer_data[['email', 'email_clean']])

# 3. Extract email domains
print("\n3. Extract email domains:")
customer_data['email_domain'] = customer_data['email_clean'].str.split('@').str[1]
print(customer_data[['email_clean', 'email_domain']])

# 4. Clean phone numbers - remove all non-digits
print("\n4. Clean phone numbers:")
customer_data['phone_clean'] = customer_data['phone'].str.replace(r'[^\d]', '', regex=True)
print(customer_data[['phone', 'phone_clean']])

# 5. Extract city and state from address
print("\n5. Extract city and state:")
# Split by comma and extract parts
customer_data['city'] = customer_data['address'].str.split(',').str[1].str.strip()
customer_data['state'] = customer_data['address'].str.split(',').str[2].str.strip()
print(customer_data[['address', 'city', 'state']])

# 6. Check for patterns
print("\n6. Pattern matching:")
# Find customers with gmail accounts
gmail_users = customer_data[customer_data['email_domain'] == 'gmail.com']
print("Gmail users:")
print(gmail_users[['name_clean', 'email_clean']])

# 7. String contains
print("\n7. Find customers in specific states:")
ny_customers = customer_data[customer_data['state'].str.contains('NY')]
print("New York customers:")
print(ny_customers[['name_clean', 'address']])

# 8. String length
print("\n8. Name lengths:")
customer_data['name_length'] = customer_data['name_clean'].str.len()
print(customer_data[['name_clean', 'name_length']])

# 9. Replace text
print("\n9. Replace text in addresses:")
customer_data['address_abbrev'] = customer_data['address'].str.replace('Street', 'St.').str.replace('Avenue', 'Ave.')
print(customer_data[['address', 'address_abbrev']].head(3))

# 10. Extract using regex patterns
print("\n10. Extract area codes from phone numbers:")
customer_data['area_code'] = customer_data['phone'].str.extract(r'(\d{3})')
print(customer_data[['phone', 'area_code']])

=== STRING PROCESSING EXAMPLES ===
Original customer data:
   customer_id            name                 email           phone  \
0         1001    John Smith    john.smith@email.com  (555) 123-4567   
1         1002        JANE DOE        JANE@GMAIL.COM    555.234.5678   
2         1003     Bob Johnson         bob@yahoo.com    555-345-6789   
3         1004     alice brown       alice@email.com   (555)456-7890   
4         1005  Charlie Wilson   charlie@outlook.com    555 567 8901   
5         1006    diana prince       diana@email.com    555.678.9012   

                          address  
0       123 Main St, New York, NY  
1         456 Oak Ave, Boston, MA  
2        789 Pine Rd, Chicago, IL  
3         321 Elm St, Houston, TX  
4       654 Maple Dr, Phoenix, AZ  
5  987 Cedar Ln, Philadelphia, PA  

1. Clean names:
             name      name_clean
0    John Smith        John Smith
1        JANE DOE        Jane Doe
2     Bob Johnson     Bob Johnson
3     alice brown     Alice Bro

### Exercise 2

Feel free to ask Gemini for help with any part of this.  If you want to check your answer, you could copy and paste any code written by Gemini into the cell below.  

You should get in the habit of inspecting what python produces and checking a few calculations by hand to ensure that what you wanted has been done.  So, try that also.

In [27]:
# Create two DataFrames to merge
products_df = pd.DataFrame({
    'product_id': [101, 102, 103, 104],
    'product_name': ['Widget A', 'Widget B', 'Widget C', 'Widget D'],
    'category': ['Electronics', 'Electronics', 'Home', 'Home']
})

sales_df = pd.DataFrame({
    'product_id': [101, 102, 101, 103, 102, 104],
    'date': pd.date_range('2024-01-01', periods=6),
    'quantity': [5, 3, 7, 2, 4, 6],
    'revenue': [50, 45, 70, 30, 60, 90]
})

# Exercise 1: Merge the products and sales DataFrames
# Use a left merge to keep all products even if they have no sales
# Assign the result to 'merged_df'


# Exercise 2: Create a pivot table showing total revenue by product_name
# Assign the result to 'revenue_pivot'


# Exercise 3: Clean the product names by making them uppercase
# Add a new column 'product_name_upper' to merged_df


# Assertions to check your answers
assert len(merged_df) == 6, "Merged DataFrame should have 6 rows"
assert 'product_name' in merged_df.columns, "Merged DataFrame should include product_name"
assert revenue_pivot.loc['Widget A'] == 120, "Widget A total revenue should be 120"
assert merged_df['product_name_upper'].iloc[0] == 'WIDGET A', "Product names should be uppercase"
print("Great job! You've successfully completed the exercises!")

NameError: name 'merged_df' is not defined

### Exercise 3

This exercise uses the famous tips dataset from seaborn (a visualization library that we will see in the next notebook).  It is a good practice dataset.

In [ ]:
# Run this cell to load and explore the tips dataset
import seaborn as sns

# Load the tips dataset
tips = sns.load_dataset('tips')

print("=== TIPS DATASET OVERVIEW ===")
print(f"Shape: {tips.shape}")
print(f"\nColumns: {list(tips.columns)}")
print(f"\nFirst 5 rows:")
print(tips.head())

print(f"\nData types:")
print(tips.dtypes)

print(f"\nBasic statistics:")
print(tips.describe())

print(f"\nUnique values in categorical columns:")
for col in ['sex', 'smoker', 'day', 'time']:
    print(f"{col}: {tips[col].unique()}")


Complete the tasks below.  Feel free to ask Gemini for help with any part of this.  

Copy the code generated by Gemini into the cell below to check your answers or inspect what the code produces on your own.

In [ ]:
# Task 1: Data Exploration
# Create a new column 'tip_percentage' that calculates tip as a percentage of total_bill
# Assign the modified DataFrame to 'tips_with_pct'


# Task 2: Grouping and Aggregation
# Calculate the average tip percentage by day of the week
# Assign the result to 'avg_tip_by_day'


# Task 3: Advanced Grouping
# Create a summary table showing average total_bill, average tip, and count of customers
# grouped by both 'time' (Lunch/Dinner) and 'smoker' (Yes/No)
# Assign the result to 'time_smoker_summary'


# Task 4: Data Filtering
# Find all the generous tippers (tip_percentage > 20%)
# Assign the filtered DataFrame to 'generous_tippers'


# Task 5: Pivot Table
# Create a pivot table showing average tip amount by day (rows) and time (columns)
# Include margins to show totals
# Assign the result to 'tip_pivot'


# Task 6: String Processing
# Add a new column 'day_type' that categorizes days as:
# - 'Weekend' for Sat and Sun
# - 'Weekday' for Thur and Fri
# Use the .str methods
# Assign the modified DataFrame to 'tips_categorized'


# Task 7: Merging Practice
# Create a small DataFrame with average meal prices by day
meal_prices = pd.DataFrame({
    'day': ['Thur', 'Fri', 'Sat', 'Sun'],
    'avg_meal_price': [25, 30, 35, 32]
})
# Merge this with tips_categorized to add the avg_meal_price column
# Assign the result to 'tips_with_prices'


# Task 8: Data Cleaning
# Create a copy of tips and randomly set 10 values to NaN in the 'tip' column
# Then fill these missing values with the median tip amount
# Assign the cleaned DataFrame to 'tips_cleaned'
import numpy as np
tips_missing = tips.copy()
np.random.seed(42)
missing_indices = np.random.choice(tips_missing.index, size=10, replace=False)
tips_missing.loc[missing_indices, 'tip'] = np.nan
# Now fill the missing values


# Assertions to check your work
assert 'tip_percentage' in tips_with_pct.columns, "Should have tip_percentage column"
assert round(tips_with_pct['tip_percentage'].mean(), 2) == 16.08, "Average tip percentage should be about 16.08%"

assert len(avg_tip_by_day) == 4, "Should have 4 days"
assert avg_tip_by_day.loc['Sun'] > 15, "Sunday average tip percentage should be > 15%"

assert time_smoker_summary.shape == (4, 3), "Summary should be 4x3 (4 groups, 3 metrics)"
assert ('Dinner', 'Yes') in time_smoker_summary.index, "Should have multi-index with time and smoker"

assert len(generous_tippers) > 30, "Should find at least 30 generous tippers"
assert all(generous_tippers['tip_percentage'] > 20), "All generous tippers should have > 20% tip"

assert 'Total' in tip_pivot.columns, "Pivot table should include margins (Total column)"
assert tip_pivot.loc['Fri', 'Lunch'] < tip_pivot.loc['Sun', 'Dinner'], "Friday lunch tips < Sunday dinner"

assert 'day_type' in tips_categorized.columns, "Should have day_type column"
assert set(tips_categorized['day_type'].unique()) == {'Weekend', 'Weekday'}, "Should have Weekend and Weekday categories"

assert 'avg_meal_price' in tips_with_prices.columns, "Should have avg_meal_price after merge"
assert len(tips_with_prices) == len(tips), "Should maintain all rows after merge"

assert tips_cleaned['tip'].isna().sum() == 0, "Should have no missing values after cleaning"
assert len(tips_cleaned) == len(tips), "Should maintain same number of rows"

print("🎉 Excellent work! You've successfully completed all the exercises!")
print("\nKey insights from your analysis:")
print(f"• Average tip percentage: {tips_with_pct['tip_percentage'].mean():.1f}%")
print(f"• Most generous day: {avg_tip_by_day.idxmax()} ({avg_tip_by_day.max():.1f}%)")
print(f"• Number of generous tippers (>20%): {len(generous_tippers)}")
print(f"• Weekend vs Weekday distribution: {tips_categorized['day_type'].value_counts().to_dict()}")

### Exercise 4: Wage Analysis

This exercise uses the wages dataset we downloaded earlier. The dataset contains information about workers' wages, education, experience, and demographics.



**Conversion of dummy variables to categorical variables**

The wages dataset accompanies a textbook on econometrics.  Dummy variables were created for several categorical variables.  This is not the usual way we encounter data.  The following cell converts the dummy variables back to categorical variables.  

It also changes 0-1 categorical variables to variables with more meaningful names, drops some variables we don't need, and creates some new categorical variables. The exercises are presented after this data conversion.

In [36]:
# Download data and create DataFrame

wages = pd.read_excel(url, header=None)
columns = [
    'wage', 'educ', 'exper',
    'tenure', 'nonwhite', 'female',
    'married', 'numdep',
    'smsa', 'northcen',
    'south', 'west', 'construc',
    'ndurman', 'trcommpu', 'trade',
    'services', 'profserv',
    'profocc', 'clerocc', 'servocc',
    'lwage', 'expersq', 'tenursq'
]
wages.columns = columns 

# Convert binary variables to categorical variables

wages['Gender'] = wages['female'].map({0: 'Male', 1: 'Female'})
wages['Race'] = wages['nonwhite'].map({0: 'White', 1: 'Non-white'})
wages['Marital_Status'] = wages['married'].map({0: 'Not Married', 1: 'Married'})
wages['Urban'] = wages['smsa'].map({0: 'Rural', 1: 'Urban'})

# Create region categories
def get_region(row):
    if row['northcen'] == 1:
        return 'North Central'
    elif row['south'] == 1:
        return 'South'
    elif row['west'] == 1:
        return 'West'
    else:
        return 'Northeast'

wages['Region'] = wages.apply(get_region, axis=1)

# Create industry categories
def get_industry(row):
    if row['construc'] == 1:
        return 'Construction'
    elif row['ndurman'] == 1:
        return 'Non-durable Manufacturing'
    elif row['trcommpu'] == 1:
        return 'Transportation/Communications'
    elif row['trade'] == 1:
        return 'Trade'
    elif row['services'] == 1:
        return 'Services'
    elif row['profserv'] == 1:
        return 'Professional Services'
    else:
        return 'Other'

wages['Industry'] = wages.apply(get_industry, axis=1)

# Create occupation categories
def get_occupation(row):
    if row['profocc'] == 1:
        return 'Professional'
    elif row['clerocc'] == 1:
        return 'Clerical'
    elif row['servocc'] == 1:
        return 'Service'
    else:
        return 'Other'

wages['Occupation'] = wages.apply(get_occupation, axis=1)

# Create education categories
wages['Education_Level'] = pd.cut(wages['educ'], 
                                 bins=[0, 12, 16, 25], 
                                 labels=['High School or Less', 'Some College', 'College Plus'])

# Create experience categories  
wages['Experience_Level'] = pd.cut(wages['exper'], 
                                  bins=[0, 5, 15, 60], 
                                  labels=['Low (0-5)', 'Mid (6-15)', 'High (16+)'])

# Drop unneeded columns
wages = wages.drop(columns=wages.columns[4:-9])

**View the modified DataFrame**

In [37]:
print("=== WAGE DATASET ANALYSIS ===")
print(f"Dataset shape: {wages.shape}")
print(f"\nFirst few rows:")
print(wages.head())
print(f"\nBasic statistics:")
print(wages.describe())

=== WAGE DATASET ANALYSIS ===
Dataset shape: (526, 13)

First few rows:
   wage  educ  exper  tenure  Gender   Race Marital_Status  Urban Region  \
0  3.10    11      2       0  Female  White    Not Married  Urban   West   
1  3.24    12     22       2  Female  White        Married  Urban   West   
2  3.00    11      2       0    Male  White    Not Married  Rural   West   
3  6.00     8     44      28    Male  White        Married  Urban   West   
4  5.30    12      7       2    Male  White        Married  Rural   West   

   Industry Occupation      Education_Level Experience_Level  
0     Other      Other  High School or Less        Low (0-5)  
1  Services    Service  High School or Less       High (16+)  
2     Trade      Other  High School or Less        Low (0-5)  
3     Other   Clerical  High School or Less       High (16+)  
4     Other      Other  High School or Less       Mid (6-15)  

Basic statistics:
             wage        educ      exper      tenure
count  526.000000  52

**Tasks**

Complete the following to analyze wage disparities and factors affecting wages.

In [ ]:
# Task 1: Statistic by group
# Calculate average wage for each gender

# Task 2: Pivot table
# Create a pivot table showing average wage by gender and marital status
# SAVE THE TABLE TO AN EXCEL FILE ON YOUR GOOGLE DRIVE

# Task 3: Statistic by multiple groups
# Create a table showing average wage by:
# - Race
# - Gender 
# - Marital status
# - Education level 
# - Experience level# Use groupby with multiple columns
# SAVE THE TABLE TO AN EXCEL FILE ON YOUR GOOGLE DRIVE


#### We've now covered the following applications of pandas.

**Data Manipulation:**

- ✅ Sorting data with `sort_values()` 
- ✅ Filtering with booleans  
- ✅ Grouping and aggregation using `groupby()` and aggregate functions  
- ✅ Data cleaning - handling missing values with `dropna()` and `fillna()` 
- ✅ Pivot tables* for data summarization and reshaping with `pivot_table()`  
- ✅ String processing with the `.str` accessor for text cleaning and extraction  

**Combining Data:**

- ✅ Merging DataFrames with `pd.merge()` using different join types (inner, left, right, outer)  
- ✅ Concatenating data vertically with `pd.concat()`  

**File Operations:**

- ✅ Reading data with `pd.read`  
- ✅ Writing data  with `df.to`  